# From AOPWiki RDF to Python objects

**Mine a schema. Generate classes. Retrieve objects.**

We will get adverse outcome pathways and the titles of their key events from [AOPWiki RDF](https://aopwiki.rdf.bigcat-bioinformatics.org/). No hand-written Python classes and no full dataset download.

Run from top to bottom with this checkout of `rdfsolve`, `pandas`, and `ipykernel` installed. The first step queries the live endpoint; results may change.

## 1. Mine the small endpoint

We select AOPWiki's data graph and skip counts and example mining. Labels and descriptions still help name the generated classes. Requests run sequentially.

In [1]:
import logging
from pathlib import Path

import pandas as pd
from rdfsolve import MinedSchema, SchemaMiner

logging.basicConfig(level=logging.WARNING, format="%(levelname)s: %(message)s", force=True)
logging.getLogger("rdfsolve.hydration").setLevel(logging.INFO)
endpoint = "https://aopwiki.rdf.bigcat-bioinformatics.org/sparql"

with SchemaMiner(
    endpoint, graph_uris=["http://aopwiki.org/"],
    strategy="one-shot", counts=False, enrich=True, examples_per_pattern=0,
    timeout=30, delay=0.5,
) as miner:
    schema = miner.mine("aopwikirdf")

print(f"{len(schema.patterns)} patterns; {len(schema.get_classes())} types")
print(f"Mining: {miner.last_report.completion_state}; {miner.last_report.total_queries_failed} failed requests")

367 patterns; 31 types
Mining: complete; 0 failed requests


## 2. Save SHACL, then generate runtime classes

We load the saved shapes to show that hydration does not depend on the miner still being present. Already have a `MinedSchema`? You can skip the file round-trip and call `schema.hydrator()` directly.

In [2]:
shapes_file = Path("aopwiki_shapes.ttl")
shapes_file.write_text(schema.to_shacl(), encoding="utf-8")
shapes = MinedSchema.from_shacl(shapes_file.read_text(encoding="utf-8"))

client = shapes.hydrator(endpoint, graph_uris=["http://aopwiki.org/"])
AOP = client.model("AdverseOutcomePathway")
print(AOP.__name__)
print(AOP.__doc__)

AdverseOutcomePathway
Observed type http://aopkb.org/aop_ontology#AdverseOutcomePathway


## 3. Get three pathways as Python objects

Choose only the fields you want. The client first selects three IRIs, then retrieves their fields together.

In [3]:
aops = client.sample(AOP, limit=3, fields=["title", "has_key_event"])

pd.DataFrame([
    {"Pathway": aop.title[0], "Key events": len(aop.has_key_event)}
    for aop in aops
])

INFO: Hydrated 3 AdverseOutcomePathway objects; 2 fields per object


,Pathway,Key events
0,Uncharacterized liver damage leading to hepato...,4
1,Binding to the picrotoxin site of ionotropic G...,6
2,Cyclooxygenase inhibition leading to reproduct...,7


These are instances of the generated class, not result dictionaries. Values are always lists: `[]` means no values were returned for a requested field. Fields you did not request stay `None`.

In [4]:
aop = aops[0]
print(type(aop).__name__)
print(aop.uri)
aop.title

AdverseOutcomePathway
https://identifiers.org/aop/1


['Uncharacterized liver damage leading to hepatocellular carcinoma']

## 4. Follow the links when you want to

Key-event references remain IRIs. Nothing downloads recursively. Here we explicitly retrieve three events, again in one batch.

In [5]:
KeyEvent = client.model("KeyEvent")
events = client.get_many(KeyEvent, aop.has_key_event[:3], fields=["title"])

pd.DataFrame([{"Event": event.title[0], "IRI": event.uri} for event in events])

INFO: Hydrated 3 KeyEvent objects; 1 fields per object


,Event,IRI
0,"Hyperplasia, Hyperplasia",https://identifiers.org/aop.events/142
1,"N/A, Unknown",https://identifiers.org/aop.events/294
2,"Promotion, Hepatocelluar carcinoma",https://identifiers.org/aop.events/334


## 5. Ask for event titles directly

Sometimes you want the answer, not the intermediate objects. Add a two-step path: **pathway → key event → title**.

Paths can come from SHACL or be supplied directly. A list of predicates means “follow these in order”.

In [6]:
AOPSummary = client.with_paths(
    AOP,
    event_titles=[
        "http://aopkb.org/aop_ontology#has_key_event",
        "http://purl.org/dc/elements/1.1/title",
    ],
)
summary = client.get(AOPSummary, aop.uri, fields=["title", "event_titles"])
summary.event_titles

INFO: Hydrated 1 AdverseOutcomePathway objects; 2 fields per object


['Hyperplasia, Hyperplasia',
 'N/A, Unknown',
 'Promotion, Hepatocelluar carcinoma',
 'Proliferation, Cell proliferation in the absence of cytotoxicity']

## 6. Keep both convenience and RDF detail

The Python fields contain usable values. Original RDF terms retain their language, datatype, and node kind separately. This matters when exporting results or writing scripts that must distinguish a literal from an IRI.

In [7]:
summary.rdf_terms["event_titles"][0]

{'kind': 'literal',
 'value': 'Hyperplasia, Hyperplasia',
 'datatype': None,
 'language': None}

## 7. Reuse the result outside the notebook

Save the generated Python classes and the hydrated object. The object includes its source graph, retrieval time, requested fields, and original RDF terms.

In [8]:
Path("aopwiki_models.py").write_text(shapes.to_pydantic(), encoding="utf-8")
Path("aop_summary.json").write_text(summary.model_dump_json(indent=2, exclude_none=True), encoding="utf-8")
print("Saved aopwiki_shapes.ttl, aopwiki_models.py, and aop_summary.json")
client.close()

Saved aopwiki_shapes.ttl, aopwiki_models.py, and aop_summary.json


## Use another source

The same client accepts a local RDFLib graph:

```python
from rdflib import Graph

data = Graph().parse("dump.ttl", format="turtle")
client = shapes.hydrator(data, graph_uris=[])
```

Or load a schema from VoID partitions with `MinedSchema.from_void(...)`. VoID metadata alone cannot describe the fields to retrieve. An RDF dump supplies the data; a mined schema or shapes supplies the view.

**Limits:** this is retrieval, not SHACL validation or OWL reasoning. Paths stay inside the selected graph. The client raises on request failures or its row limit; it cannot detect every hidden endpoint cap. Blank nodes are retained as references scoped to a response, not followed across requests. No automatic recursive hydration or background requests.